# Jour 1 · Découvrir une série temporelle


## Objectifs

- reconnaître la structure d'un jeu de données IoT
- convertir une colonne en timestamp exploitable
- trier, indexer et visualiser les mesures dans le temps

Une série temporelle est une suite de mesures **ordonnées dans le temps**. L'ordre n'est pas un détail : il raconte ce qui s'est passé avant et après.

Dans ce fil rouge, un équipement HVAC envoie température, humidité, puissance, pression et vibration toutes les quinze minutes.

## Avant de coder — comprendre le tableau

Le fichier CSV peut être imaginé comme un tableau :

| Mot | Sens dans notre fichier | Exemple |
|---|---|---|
| Ligne / observation | un message envoyé par le capteur à un instant | mesures de `hvac_01` à 10 h 15 |
| Colonne / variable | une information présente dans chaque message | `temperature_c` |
| Valeur | le contenu d'une case | `21.4` |
| Type | la nature de la valeur | texte, nombre, date… |
| Valeur manquante | une case existe mais son contenu manque | `NaN` |

Pandas charge ce tableau dans un **DataFrame**. Un DataFrame n'est donc pas un modèle de machine learning : c'est d'abord une structure pratique pour inspecter et transformer des données.

### Les premières questions à poser à tout nouveau fichier

1. Qu'est-ce qu'une ligne représente ?
2. Quelles sont les colonnes et leurs unités ?
3. Combien y a-t-il de lignes ?
4. Quels types Pandas a-t-il reconnus ?
5. Certaines valeurs ou certains instants manquent-ils ?
6. Les lignes sont-elles rangées dans l'ordre du temps ?

![Même série temporelle présentée en tableau puis sous forme de courbe](../assets/jour_01/01_serie_temporelle_table_et_courbe.png)

*Chaque point de la courbe correspond à une ligne du tableau : une valeur, un timestamp, puis l'ordre du temps qui les relie.*

## Charger le CSV

`import pandas as pd` rend la bibliothèque Pandas disponible sous le nom court `pd`. `read_csv(...)` lit le fichier et renvoie un DataFrame.

La fonction `find_project_root()` ci-dessous sert uniquement à retrouver le dossier `datasets`, même si Jupyter a été lancé depuis un sous-dossier. Vous n'avez pas besoin de la mémoriser : concentrez-vous pour l'instant sur `pd.read_csv(...)`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Dossier datasets introuvable. Lancez Jupyter depuis le projet.")


ROOT = find_project_root()
DATA_DIR = ROOT / "datasets"


plt.style.use("seaborn-v0_8-whitegrid")
raw = pd.read_csv(DATA_DIR / "raw" / "iot_hvac_raw.csv")
print(f"Lignes : {len(raw):,} | Colonnes : {raw.shape[1]}")
raw.head()

## Lire cette première sortie

- `len(raw)` donne le nombre de lignes.
- `raw.shape` donne `(nombre de lignes, nombre de colonnes)`.
- `raw.head()` affiche les cinq premières lignes **dans l'ordre où elles se trouvent dans le fichier**.

Attention : « premières lignes du fichier » ne signifie pas encore « premières mesures dans le temps ». Le fichier brut a volontairement été désordonné pour reproduire des messages reçus en retard.

Les noms comme `temperature_c` et `power_kw` contiennent l'unité. C'est précieux : une valeur `22` est ambiguë si l'on ignore s'il s'agit de degrés Celsius, de volts ou de kilowatts.

In [ ]:
print(raw.dtypes)
print("\nValeurs manquantes :")
print(raw.isna().sum())

## Types et valeurs manquantes

`dtypes` indique comment Pandas interprète chaque colonne :

| Type affiché | Sens simple |
|---|---|
| `object` | souvent du texte |
| `float64` | nombre décimal ; peut aussi contenir `NaN` |
| `int64` | nombre entier |
| `datetime64[...]` | date et heure reconnues comme telles |

`isna()` répond vrai ou faux pour chaque case, puis `sum()` compte les valeurs vraies colonne par colonne. `NaN` signifie qu'une **case** est vide. Nous verrons plus tard qu'un message totalement absent est différent : dans ce cas, la ligne elle-même n'existe pas encore.

Ici, `timestamp` est encore du texte (`object`). Tant qu'il reste du texte, Pandas ne peut pas calculer correctement une durée, regrouper par heure ou sélectionner une période.

## Transformer le texte en véritable date

`pd.to_datetime(...)` convertit le texte en dates exploitables :

- `utc=True` normalise les instants en UTC, ce qui évite les ambiguïtés entre sites et changements d'heure ;
- `errors="coerce"` remplace une date illisible par `NaT` plutôt que d'arrêter tout le notebook ;
- `NaT` signifie **Not a Time**, l'équivalent temporel de `NaN`.

L'UTC est adapté au stockage. Pour un tableau de bord destiné à un technicien, on peut ensuite convertir l'affichage dans son heure locale.

In [ ]:
df = raw.copy()
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

print("Timestamps invalides :", df["timestamp"].isna().sum())
print("Déjà trié ?", df["timestamp"].is_monotonic_increasing)
print("Début :", df["timestamp"].min())
print("Fin   :", df["timestamp"].max())

## Comment lire le diagnostic ?

- `isna().sum()` compte les dates qui n'ont pas pu être converties.
- `is_monotonic_increasing` vaut `True` uniquement si chaque date est supérieure ou égale à la précédente.
- `min()` et `max()` donnent les bornes temporelles.

Un fichier peut contenir des dates valides tout en étant désordonné. Avant une différence entre deux lignes, une moyenne mobile ou un découpage passé/futur, il faut donc **trier explicitement**.

Un timestamp dupliqué signifie que plusieurs lignes revendiquent le même instant. Ce n'est pas automatiquement une erreur : plusieurs équipements pourraient mesurer au même instant. Dans notre fichier, qui ne contient qu'un équipement, ce doublon doit être examiné.

### À vous de jouer — faire le premier diagnostic

Affichez le nombre de timestamps dupliqués, le nombre de capteurs et la durée totale couverte par le fichier.

**Indice :** Utilisez duplicated(), nunique(), min() et max().

In [ ]:
# Complétez une ligne à la fois à l'aide de l'indice.
# duplicate_count = df["timestamp"].____().____()
# device_count = df["device_id"].____()
# duration = df["timestamp"].____() - df["timestamp"].____()
pass

## Trier puis choisir un index temporel

`sort_values("timestamp")` remet les observations dans l'ordre chronologique. `set_index("timestamp")` place ensuite la date dans l'index, c'est-à-dire dans l'étiquette de chaque ligne.

```text
Avant                         Après set_index
ligne  timestamp   temp       timestamp          temp
0      10:15       21.4       10:00              21.2
1      10:00       21.2       10:15              21.4
```

Ce choix permet ensuite d'écrire `ordered.loc[début:fin]`, de rééchantillonner ou de calculer des fenêtres temporelles. `loc` inclut ici les deux bornes.

![Comparaison d'une courbe désordonnée et de la même courbe triée chronologiquement](../assets/jour_01/01_ordre_temporel.png)

*Relier les lignes dans leur ordre d'arrivée fabrique une fausse histoire ; le tri chronologique restitue l'évolution réelle.*

In [ ]:
ordered = df.sort_values("timestamp").set_index("timestamp")
window = ordered.loc[ordered.index.min(): ordered.index.min() + pd.Timedelta(days=3)]

ax = window["temperature_c"].plot(figsize=(12, 4), title="Température — trois premiers jours")
ax.set_ylabel("°C")
plt.show()

## Lire un graphique temporel

L'axe horizontal représente le temps et doit rester dans l'ordre. L'axe vertical représente la valeur et son unité.

Avant de chercher une anomalie, observez toujours :

- le **niveau habituel** du signal ;
- son amplitude minimale et maximale ;
- les répétitions visibles ;
- les changements brusques ;
- les périodes sans trait ou avec des valeurs manquantes.

Un graphique attire l'attention, mais ne prouve pas une cause. Une hausse de puissance peut être une panne, une mise en route normale ou un changement d'activité.

### À vous de jouer — visualiser deux signaux

Sur les deux premiers jours, tracez l'humidité puis la puissance dans deux sous-graphiques partageant le même axe temporel.

**Indice :** Sélectionnez les deux colonnes puis utilisez plot(subplots=True).

In [ ]:
# 1. Sélectionnez les deux premiers jours avec ordered.loc[...].
# 2. Gardez les colonnes humidity_pct et power_kw.
# 3. Appelez .plot(subplots=True, sharex=True).
pass

## À retenir

- Une date stockée comme texte doit être convertie avant toute analyse.
- Une série temporelle doit être triée avant les calculs dépendant de l'ordre.
- Stocker les instants en UTC évite de nombreuses ambiguïtés ; la conversion locale sert surtout à l'affichage.

### Mini-mémo

| Besoin | Commande |
|---|---|
| Charger un CSV | `pd.read_csv(...)` |
| Voir les premières lignes | `df.head()` |
| Connaître la taille | `df.shape` |
| Inspecter les types | `df.dtypes` |
| Compter les cases vides | `df.isna().sum()` |
| Convertir une date | `pd.to_datetime(...)` |
| Trier chronologiquement | `df.sort_values("timestamp")` |
| Utiliser la date comme index | `df.set_index("timestamp")` |